# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davidsoooooon/ML-Intern---Briones/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### My method choice

My task is a ranking problem supported by a binary proxy: whether a page's impressions decline by more than 20% in the later 14-day window.

I will start with Logistic Regression because it provides a simple and readable comparison. I will also test a Random Forest to see whether nonlinear relationships between early impressions, search position, position volatility, and active days improve the ranking.

Both models will produce probabilities rather than only yes-or-no predictions. I will use those probabilities to rank pages for editorial review and evaluate them mainly with Precision@50.

I will only treat the more complex model as useful if it performs better than both Logistic Regression and my Week-4 baseline on the same client-held-out test set.

In [6]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

model_data = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN gsc_impressions ELSE 0
            END) AS impressions_early,

            AVG(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN NULLIF(gsc_avg_position, 0)
            END) AS avg_position_early,

            STDDEV_SAMP(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                THEN NULLIF(gsc_avg_position, 0)
            END) AS position_volatility_early,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-14'
                     AND gsc_impressions > 0
                THEN report_date
            END) AS active_days_early,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-15'
                                     AND DATE '2026-03-28'
                THEN gsc_impressions ELSE 0
            END) AS impressions_late

        FROM {MARCH_DAILY}
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-28'
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,
        CASE
            WHEN impressions_late < 0.80 * impressions_early
            THEN 1 ELSE 0
        END AS is_declining_proxy
    FROM page_windows
    WHERE impressions_early >= 100
""").df()

FEATURES = [
    "impressions_early",
    "avg_position_early",
    "position_volatility_early",
    "active_days_early"
]

TARGET = "is_declining_proxy"
GROUP = "client_hash_id"

FORBIDDEN_FEATURES = {
    "impressions_late",
    "is_declining_proxy"
}

assert set(FEATURES).isdisjoint(FORBIDDEN_FEATURES)

print(f"Pages: {len(model_data):,}")
print(f"Clients: {model_data[GROUP].nunique():,}")
print(f"Declining rate: {model_data[TARGET].mean():.3f}")

display(model_data[FEATURES + [TARGET]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 75,526
Clients: 37
Declining rate: 0.338


,impressions_early,avg_position_early,position_volatility_early,active_days_early,is_declining_proxy
0,211.0,4.245075,2.342553,14,0
1,1394.0,6.104078,1.173984,13,0
2,237.0,4.542255,1.455799,14,1
3,385.0,4.351876,1.405632,14,0
4,272.0,10.398054,4.968515,14,1


## 2. Split design

### My split design

I use a client-grouped holdout split. Around 80% of the clients are used for training and the remaining 20% are used for testing.

All pages from one client stay entirely in either the training set or the test set. This is more honest than randomly splitting individual pages because pages belonging to the same client may share similar search patterns.

The features use information from March 1–14, while the proxy outcome uses impressions from March 15–28. This keeps the feature window earlier than the outcome window.

I fixed the random seed at 42 so the split and results can be reproduced. The baseline and both models will be evaluated using the same held-out test pages. This is still a single holdout experiment, so I treat the findings as directional rather than final production evidence.

In [7]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        model_data[FEATURES],
        model_data[TARGET],
        groups=model_data[GROUP]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()

X_train = train_data[FEATURES]
X_test = test_data[FEATURES]

y_train = train_data[TARGET].astype(int)
y_test = test_data[TARGET].astype(int)

train_clients = set(train_data[GROUP])
test_clients = set(test_data[GROUP])

assert train_clients.isdisjoint(test_clients)
assert y_train.nunique() == 2
assert y_test.nunique() == 2

print(f"Training pages: {len(train_data):,}")
print(f"Testing pages: {len(test_data):,}")
print(f"Training clients: {len(train_clients)}")
print(f"Testing clients: {len(test_clients)}")
print(f"Training declining rate: {y_train.mean():.3f}")
print(f"Testing declining rate: {y_test.mean():.3f}")
print(f"Shared clients: {len(train_clients & test_clients)}")

Training pages: 65,274
Testing pages: 10,252
Training clients: 29
Testing clients: 8
Training declining rate: 0.349
Testing declining rate: 0.265
Shared clients: 0


## 3. Train + compare vs my baseline

### Training and comparison

I rebuilt my Week-4 rule using only the training data. The 75th-percentile thresholds and percentile strengths are learned from the training clients, then applied to the held-out test clients.

I trained Logistic Regression as the readable first model and Random Forest as the more flexible comparison. Both models rank pages using their predicted probabilities.

The table compares every method on the same test pages using Precision@20, Precision@50, average precision, and ROC-AUC. Precision@50 remains my primary metric because the intended action is a limited editorial review queue.

In [8]:
def precision_at_k(y_true, scores, k):
    ranked = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    }).sort_values("score", ascending=False)

    return ranked.head(k)["actual"].mean()


def percentile_from_train(train_series, new_series):
    train_values = (
        pd.to_numeric(train_series, errors="coerce")
        .dropna()
        .sort_values()
        .to_numpy()
    )

    new_values = (
        pd.to_numeric(new_series, errors="coerce")
        .fillna(-np.inf)
        .to_numpy()
    )

    return (
        np.searchsorted(train_values, new_values, side="right")
        / len(train_values)
    )


# Rebuild the Week-4 baseline using training data only
volume_cutoff = train_data["impressions_early"].quantile(0.75)

volatility_cutoff = train_data[
    "position_volatility_early"
].quantile(0.75)

test_volume_strength = percentile_from_train(
    train_data["impressions_early"],
    test_data["impressions_early"]
)

test_volatility_strength = percentile_from_train(
    train_data["position_volatility_early"],
    test_data["position_volatility_early"]
)

baseline_rule_hit = (
    (test_data["impressions_early"] >= volume_cutoff)
    & (
        test_data["position_volatility_early"]
        >= volatility_cutoff
    )
)

baseline_test_scores = np.where(
    baseline_rule_hit,
    test_volume_strength * test_volatility_strength,
    0.0
)


# Define the models
models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=RANDOM_STATE
        )
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
}


# Train models and collect test scores
fitted_models = {}
test_scores = {
    "Week-4 baseline rule": baseline_test_scores
}

for name, model in models.items():
    model.fit(X_train, y_train)

    fitted_models[name] = model
    test_scores[name] = model.predict_proba(X_test)[:, 1]


# Build the comparison table
results = []

for name, scores in test_scores.items():
    results.append({
        "Method": name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Average precision": average_precision_score(y_test, scores),
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Test base rate": y_test.mean()
    })

comparison = (
    pd.DataFrame(results)
    .sort_values(
        ["Precision@50", "Average precision"],
        ascending=False
    )
    .reset_index(drop=True)
)

best_model_name = max(
    fitted_models,
    key=lambda name: precision_at_k(
        y_test,
        test_scores[name],
        50
    )
)

print(f"Training volume cutoff: {volume_cutoff:,.2f}")
print(f"Training volatility cutoff: {volatility_cutoff:.2f}")
print(f"Baseline pages flagged in test set: {baseline_rule_hit.sum():,}")
print(f"Best learned model by Precision@50: {best_model_name}")

display(
    comparison.style.format({
        "Precision@20": "{:.3f}",
        "Precision@50": "{:.3f}",
        "Average precision": "{:.3f}",
        "ROC-AUC": "{:.3f}",
        "Test base rate": "{:.3f}"
    })
)

Training volume cutoff: 1,612.00
Training volatility cutoff: 6.33
Baseline pages flagged in test set: 71
Best learned model by Precision@50: Logistic Regression


,Method,Precision@20,Precision@50,Average precision,ROC-AUC,Test base rate
0,Logistic Regression,0.500,0.480,0.258,0.480,0.265
1,Random Forest,0.200,0.280,0.289,0.536,0.265
2,Week-4 baseline rule,0.350,0.240,0.265,0.500,0.265


## 4. Errors and interpretation

### Error analysis approach

Because the model supports a top-50 editorial queue, I treat non-declining pages inside its top 50 as its most relevant errors.

I inspect three concrete wrong picks, compare errors across early-impression and position-volatility ranges, and use held-out permutation importance to see which features the selected model relies on.

Permutation importance is measured with average precision. Positive values indicate that shuffling the feature reduced performance, while values close to zero or below zero suggest that the feature was not consistently helpful on the held-out clients.

In [9]:
best_model = fitted_models[best_model_name]
best_scores = test_scores[best_model_name]

review = test_data[
    ["content_hash_id"] + FEATURES + [TARGET]
].copy()

review["model_score"] = best_scores

review["impression_band"] = pd.qcut(
    review["impressions_early"],
    q=4,
    duplicates="drop"
)

review["volatility_band"] = pd.qcut(
    review["position_volatility_early"],
    q=4,
    duplicates="drop"
)

review = review.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

review["model_rank"] = np.arange(1, len(review) + 1)

top50 = review.head(50).copy()
wrong_picks = top50[top50[TARGET] == 0].copy()

print(f"Correct declining pages in top 50: {top50[TARGET].sum()}")
print(f"Wrong non-declining pages in top 50: {len(wrong_picks)}")
print(f"Top-50 error rate: {1 - top50[TARGET].mean():.3f}")

print("\nThree concrete wrong picks:")

display(
    wrong_picks[
        [
            "model_rank",
            "content_hash_id",
            "model_score",
            "impressions_early",
            "avg_position_early",
            "position_volatility_early",
            "active_days_early"
        ]
    ].head(3)
)


def top50_error_table(frame, band_column):
    return (
        frame.groupby(band_column, observed=True)
        .agg(
            reviewed_pages=(TARGET, "size"),
            correct_declines=(TARGET, "sum"),
            precision=(TARGET, "mean")
        )
        .assign(
            wrong_picks=lambda table:
                table["reviewed_pages"] - table["correct_declines"]
        )
        .reset_index()
    )


print("Top-50 results by early-impression range:")

display(
    top50_error_table(
        top50,
        "impression_band"
    ).style.format({"precision": "{:.3f}"})
)

print("Top-50 results by position-volatility range:")

display(
    top50_error_table(
        top50,
        "volatility_band"
    ).style.format({"precision": "{:.3f}"})
)


# Held-out permutation importance
permutation = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_table = (
    pd.DataFrame({
        "Feature": FEATURES,
        "Importance mean": permutation.importances_mean,
        "Importance std": permutation.importances_std
    })
    .sort_values("Importance mean", ascending=False)
    .reset_index(drop=True)
)

print("Held-out permutation importance:")

display(
    importance_table.style.format({
        "Importance mean": "{:.4f}",
        "Importance std": "{:.4f}"
    })
)

Correct declining pages in top 50: 24
Wrong non-declining pages in top 50: 26
Top-50 error rate: 0.520

Three concrete wrong picks:


,model_rank,content_hash_id,model_score,impressions_early,avg_position_early,position_volatility_early,active_days_early
0,1,content_1c47c13983830602,0.712185,9678.0,59.674797,2.815592,14
3,4,content_bfb22aff4e9340f6,0.690285,791.0,59.766097,4.722032,14
5,6,content_76d99ffa3d1b90d1,0.681601,483.0,56.269803,3.961511,14


Top-50 results by early-impression range:


,impression_band,reviewed_pages,correct_declines,precision,wrong_picks
0,"(99.999, 198.0]",8,3,0.375,5
1,"(198.0, 407.0]",6,2,0.333,4
2,"(407.0, 908.0]",16,10,0.625,6
3,"(908.0, 49127.0]",20,9,0.450,11


Top-50 results by position-volatility range:


,volatility_band,reviewed_pages,correct_declines,precision,wrong_picks
0,"(1.705, 3.305]",9,3,0.333,6
1,"(3.305, 5.957]",22,14,0.636,8
2,"(5.957, 33.656]",19,7,0.368,12


Held-out permutation importance:


,Feature,Importance mean,Importance std
0,active_days_early,0.0073,0.0019
1,impressions_early,-0.0032,0.0007
2,position_volatility_early,-0.0055,0.0019
3,avg_position_early,-0.0121,0.0042


### What the results mean

On the held-out clients, Logistic Regression achieved a Precision@50 of 0.480, compared with 0.240 for my Week-4 baseline. This means the model found 24 declining pages in its top 50, while the baseline found 12. That is an absolute improvement of 0.240 and twice as many useful picks within the review limit.

Logistic Regression also improved Precision@20 from 0.350 to 0.500. Random Forest reached only 0.280 at Precision@50, so its additional complexity was not justified for this review-queue objective.

However, the Logistic Regression result is not strong across the entire test set. Its average precision was 0.258, slightly below the test base rate of 0.265, and its ROC-AUC was 0.480. This suggests that its improvement is concentrated near the top of the queue rather than representing a reliable ranking of all pages. I therefore treat the result as directional decision support, not production-ready evidence.

### What the errors look like

The model made 26 wrong picks in its top 50, giving it a top-50 error rate of 0.520. The strongest impression range was 407–908 impressions, where precision reached 0.625. The 198–407 range was the weakest at 0.333, while the highest-volume range reached only 0.450. This shows that more impressions did not consistently mean a higher chance of decline.

The middle volatility range of 3.31–5.96 performed best, with precision of 0.636. The highest-volatility range fell to 0.368 and contained 12 wrong picks. High volatility may sometimes represent normal ranking noise, seasonality, or mixed search queries rather than a content problem.

The three highest-ranked wrong cases all had average positions around 56–60 and were active throughout the 14-day feature window. Their deep rankings may have looked risky to the model, but they did not experience the later decline represented by the proxy.

### Feature interpretation

Permutation importance provided weak evidence of stable feature relationships. `active_days_early` was the only feature with positive importance, at 0.0073. This is plausible because consistent activity gives the model more observed evidence than a page appearing on only a few days.

The other features had negative importance, meaning that shuffling them slightly improved held-out average precision. This does not prove that impressions, position, or volatility are useless. It suggests that the relationships learned from the training clients did not transfer consistently to the held-out clients.

Overall, Logistic Regression beat the baseline at the specific top-50 decision point, but the errors and weak global metrics show that it should help editors prioritize investigation—not automatically decide which pages require changes.

In [10]:
import sklearn

print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Random seed: {RANDOM_STATE}")

pandas version: 2.2.2
scikit-learn version: 1.6.1
DuckDB version: 1.3.2
Random seed: 42


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.